# OCR / VLM Smoke Test

This notebook is a lightweight smoke harness for testing OCR or multimodal LLM models on Kaggle.

It uses fixed benchmark v2 pages directly from the repo, so image paths and GT markdown are resolved automatically.

Selected pages:
- `MWG_Q2_2023_p008`
- `TCB_Q3_2024_p006`
- `VCB_Q1_2022_p004`
- `VIC_Q4_2021_p009`


In [ ]:
# Optional installs. Uncomment what you need for the model you want to test.
# %pip install -q openai pillow ipython jiwer
# %pip install -q transformers accelerate
# %pip install -q llama-cpp-python


In [ ]:
from __future__ import annotations

import base64
import json
import mimetypes
import subprocess
import sys
from pathlib import Path
from typing import Callable, Dict, List

from IPython.display import Markdown, Image as IPyImage, display

OUTPUT_DIR = Path("results/smoke_test_markdown")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REFERENCE_DPI = 300
TEMPERATURE = 0.0
TOP_P = 1.0
MAX_OUTPUT_TOKENS = 8192
SAVE_MARKDOWN = True
SAVE_METRICS_JSON = True
PRETTY_RENDER_MARKDOWN = True
MODEL_NAME = "set-me"

BENCH_REPO_URL = "https://github.com/buinguyenkhai/stock-report-agent-20251.git"
BENCH_REPO_DIR = Path("/kaggle/working/stock-report-agent-20251")
AUTO_CLONE_BENCH_REPO = True
DATASET_ROOT_REL = Path("data/benchmark_v2")
SAMPLE_IDS = [
    "MWG_Q2_2023_p008",
    "TCB_Q3_2024_p006",
    "VCB_Q1_2022_p004",
    "VIC_Q4_2021_p009",
]

OCR_PROMPT = """
Transcribe this financial-report page into Markdown.

Rules:
1. Return only the page content in Markdown. Do not add explanations, summaries, or code fences.
2. Focus on the visible table content on the page. If multiple tables are visible, output them in page order, separated by a blank line.
3. Preserve the original language, row order, column order, and multi-row headers as faithfully as possible.
4. Keep numbers exactly as shown, including decimal separators, thousand separators, parentheses, minus signs, percent signs, and unit markers.
5. Do not normalize, translate, infer, or correct values.
6. Prefer Markdown pipe tables whenever the table structure is visible.
7. If a header line or unit line clearly belongs to the table, keep it immediately above or inside the table in Markdown.
8. If some text is unclear, transcribe best effort rather than inventing missing content.
""".strip()

print(f"Reference DPI: {REFERENCE_DPI}")
print(f"Output dir: {OUTPUT_DIR.resolve()}")
print(f"Model name: {MODEL_NAME}")


In [ ]:
def ensure_benchmark_repo() -> Path:
    if BENCH_REPO_DIR.exists():
        return BENCH_REPO_DIR
    if not AUTO_CLONE_BENCH_REPO:
        raise FileNotFoundError(f"Benchmark repo not found at {BENCH_REPO_DIR}")
    subprocess.run(["git", "clone", BENCH_REPO_URL, str(BENCH_REPO_DIR)], check=True)
    return BENCH_REPO_DIR


BENCH_REPO = ensure_benchmark_repo()
DATASET_ROOT = BENCH_REPO / DATASET_ROOT_REL
if str(BENCH_REPO) not in sys.path:
    sys.path.insert(0, str(BENCH_REPO))

from evaluation.benchmark_v2.metrics_raw import calculate_raw_metrics

manifest = json.loads((DATASET_ROOT / "manifest.json").read_text(encoding="utf-8"))
sample_map = {row["sample_id"]: row for row in manifest["samples"]}

IMAGE_SPECS = []
for sample_id in SAMPLE_IDS:
    row = sample_map[sample_id]
    IMAGE_SPECS.append(
        {
            "alias": row["company"].lower(),
            "sample_id": sample_id,
            "image_name": Path(row["page_image_path"]).name,
            "image_path": DATASET_ROOT / row["page_image_path"],
            "gt_markdown_path": DATASET_ROOT / row["gt_markdown_path"],
        }
    )

print(f"Imported benchmark metrics from: {BENCH_REPO}")
for spec in IMAGE_SPECS:
    print(spec["sample_id"], "->", spec["image_path"])


In [ ]:
def image_to_data_url(image_path: Path) -> str:
    mime = mimetypes.guess_type(str(image_path))[0] or "image/png"
    encoded = base64.b64encode(image_path.read_bytes()).decode("utf-8")
    return f"data:{mime};base64,{encoded}"


def preview_images(image_specs: List[dict]) -> None:
    for spec in image_specs:
        print(f"{spec['sample_id']}: {spec['image_path']}")
        display(IPyImage(filename=str(spec["image_path"]), width=900))


preview_images(IMAGE_SPECS)


In [ ]:
def build_openai_compatible_messages(prompt: str, image_path: Path) -> List[dict]:
    return [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": image_to_data_url(image_path)}},
            ],
        }
    ]


def run_openai_compatible(client, *, model: str, image_path: Path, prompt: str) -> str:
    response = client.chat.completions.create(
        model=model,
        messages=build_openai_compatible_messages(prompt, image_path),
        temperature=TEMPERATURE,
        top_p=TOP_P,
        max_tokens=MAX_OUTPUT_TOKENS,
    )
    content = response.choices[0].message.content
    if isinstance(content, list):
        return "\n".join(part.get("text", "") for part in content if isinstance(part, dict)).strip()
    return str(content or "").strip()


def save_markdown(model_name: str, image_name: str, markdown_text: str) -> Path:
    safe_model_name = model_name.replace('/', '__').replace(':', '_')
    out_dir = OUTPUT_DIR / safe_model_name
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{Path(image_name).stem}.md"
    out_path.write_text(markdown_text, encoding="utf-8")
    return out_path


def save_metrics(model_name: str, metrics_by_image: Dict[str, dict]) -> Path:
    safe_model_name = model_name.replace('/', '__').replace(':', '_')
    out_dir = OUTPUT_DIR / safe_model_name
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / "metrics.json"
    out_path.write_text(json.dumps(metrics_by_image, ensure_ascii=False, indent=2), encoding="utf-8")
    return out_path


def render_markdown(title: str, markdown_text: str) -> None:
    print(title)
    if PRETTY_RENDER_MARKDOWN:
        display(Markdown(markdown_text))
    else:
        print(markdown_text)


def print_metric_summary(alias: str, sample_id: str, metrics: dict) -> None:
    print(
        f"[{alias} | {sample_id}] CER={metrics['table_only_cer']:.4f} "
        f"WER={metrics['table_only_wer']:.4f} "
        f"CellF1={metrics['table_cell_f1']:.4f} "
        f"NumF1={metrics['number_f1']:.4f}"
    )


def run_batch(image_specs: List[dict], infer_fn: Callable[[Path], str], *, model_name: str) -> Dict[str, dict]:
    outputs: Dict[str, dict] = {}
    metrics_by_image: Dict[str, dict] = {}
    for spec in image_specs:
        alias = spec['alias']
        sample_id = spec['sample_id']
        image_name = spec['image_name']
        image_path = spec['image_path']
        markdown_text = infer_fn(image_path)
        gt_markdown = spec['gt_markdown_path'].read_text(encoding='utf-8')
        metrics = calculate_raw_metrics(markdown_text, gt_markdown).to_dict()
        record = {
            'alias': alias,
            'sample_id': sample_id,
            'image_name': image_name,
            'image_path': str(image_path),
            'gt_markdown_path': str(spec['gt_markdown_path']),
            'markdown': markdown_text,
            'metrics': metrics,
        }
        outputs[alias] = record
        metrics_by_image[alias] = metrics
        if SAVE_MARKDOWN:
            saved = save_markdown(model_name, image_name, markdown_text)
            record['markdown_path'] = str(saved)
            print(f"Saved: {saved}")
        print_metric_summary(alias, sample_id, metrics)
        render_markdown(f"## {sample_id}", markdown_text)
    if SAVE_METRICS_JSON:
        metrics_path = save_metrics(model_name, metrics_by_image)
        print(f"Saved metrics: {metrics_path}")
    return outputs


def summarize_batch(outputs: Dict[str, dict]) -> None:
    scored = [row for row in outputs.values() if row.get('metrics')]
    if not scored:
        print('No scored outputs yet.')
        return
    metric_keys = ['table_only_cer', 'table_only_wer', 'table_cell_f1', 'number_f1']
    print('\nBenchmark v2 raw-metric summary')
    for key in metric_keys:
        avg = sum(float(row['metrics'][key]) for row in scored) / len(scored)
        print(f" - {key}: {avg:.4f}")


In [ ]:
# Example 1: OpenAI-compatible multimodal API
# from openai import OpenAI
# client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])
# MODEL_NAME = "your-vision-model"
# outputs = run_batch(
#     IMAGE_SPECS,
#     lambda image_path: run_openai_compatible(client, model=MODEL_NAME, image_path=image_path, prompt=OCR_PROMPT),
#     model_name=MODEL_NAME,
# )
# summarize_batch(outputs)

# Example 2: llama-cpp-python vision model with local image data URL
# from llama_cpp import Llama
# llm = Llama.from_pretrained(repo_id="your/repo", filename="your-model.gguf")
# MODEL_NAME = "your-gguf-vision-model"
# def llama_cpp_infer(image_path: Path) -> str:
#     response = llm.create_chat_completion(
#         messages=build_openai_compatible_messages(OCR_PROMPT, image_path),
#         temperature=TEMPERATURE,
#         top_p=TOP_P,
#         max_tokens=MAX_OUTPUT_TOKENS,
#     )
#     return response["choices"][0]["message"]["content"].strip()
# outputs = run_batch(IMAGE_SPECS, llama_cpp_infer, model_name=MODEL_NAME)
# summarize_batch(outputs)

# Example 3: Custom model hook
# MODEL_NAME = "custom-model"
# def custom_infer(image_path: Path) -> str:
#     raise NotImplementedError
# outputs = run_batch(IMAGE_SPECS, custom_infer, model_name=MODEL_NAME)
# summarize_batch(outputs)


In [ ]:
# Optional: inspect saved artifacts after running one of the examples above.
saved_files = sorted(OUTPUT_DIR.rglob('*'))
print(f"Saved artifacts: {len(saved_files)}")
for path in saved_files:
    if path.is_file():
        print(path)
